In [0]:
# display(spark.sql("SELECT * FROM samples.wanderbricks.users"))÷

from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from delta.tables import DeltaTable

# -----------------------------
# 1. Get vars (widgets + defaults)
# -----------------------------

def get_widget_or_default(name: str, default: str) -> str:
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

DEFAULT_CATALOG = "eligibility_operation"
DEFAULT_SCHEMA = "default"
DEFAULT_STAGING_TABLE = "member_eligibility_staging"
DEFAULT_REJECTED_TABLE = "member_eligibility_rejected"
TARGET_TABLE = "member_eligibility"

catalog = get_widget_or_default("catalog", DEFAULT_CATALOG)
schema = get_widget_or_default("schema", DEFAULT_SCHEMA)
staging_table_name = get_widget_or_default("staging_table", DEFAULT_STAGING_TABLE)
rejected_table_name = get_widget_or_default("rejected_table", DEFAULT_REJECTED_TABLE)

db = f"{catalog}.{schema}"
staging_table = f"{db}.{staging_table_name}"
rejected_table = f"{db}.{rejected_table_name}"
target_table = f"{db}.{TARGET_TABLE}"

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

# -----------------------------
# 2. Read from staging (PENDING only for idempotency)
# -----------------------------

staging_df = (
    spark.table(staging_table)
)

clean_records = staging_df.filter(col("validation_errors").isNull())

clean_records = clean_records.withColumn("dob", to_date(col("dob"), "yyyy-MM-dd")).withColumn("load_date", current_timestamp())

target_columns = spark.table(target_table).columns

clean_records = clean_records.select(*target_columns)

clean_records.write.mode("append").saveAsTable(target_table)


_1
external_id
first_name
last_name
dob
email
phone
partner_code
load_date
partner_id


staging_id,partner_id,partner_code,external_id,first_name,last_name,dob,email,phone,validation_status,validation_errors,created_date,load_date
21,2,bettercare,BC-001,Alice,Johnson,1965-08-10,alice.j@test.com,5552223333,FINISHED,null,null,2026-01-15T22:40:46.159Z
22,2,bettercare,BC-002,Charlie,Brown,1972-03-25,charlie.b@test.com,5554445555,FINISHED,null,null,2026-01-15T22:40:46.159Z
25,2,bettercare,BC-005,FRANK,LOPEZ,1945-03-30,FRANK.LOPEZ@TEST.COM,+15551112222,FINISHED,null,null,2026-01-15T22:40:46.159Z
26,2,bettercare,BC-006,"Maria, Jr.",Rodriguez,1982-07-25,maria.rodriguez@health.org,5553334444,FINISHED,null,null,2026-01-15T22:40:46.159Z
28,2,bettercare,BC-008,Hannah,Anderson,1995-11-18,hannah.anderson@example.com,5557778888,FINISHED,null,null,2026-01-15T22:40:46.159Z
29,2,bettercare,BC-001,Isaac,Thompson,2025-12-31,isaac.thompson@test.com,(555) 999-0000,FINISHED,null,null,2026-01-15T22:40:46.159Z
30,2,bettercare,BC-010,Julia,White,1963-04-22,julia.white@mail.com,5552221111,FINISHED,null,null,2026-01-15T22:40:46.159Z
11,2,bettercare,BC-001,Alice,Johnson,1965-08-10,alice.j@test.com,5552223333,FINISHED,null,null,2026-01-15T22:40:46.159Z
12,2,bettercare,BC-002,Charlie,Brown,1972-03-25,charlie.b@test.com,5554445555,FINISHED,null,null,2026-01-15T22:40:46.159Z
15,2,bettercare,BC-005,FRANK,LOPEZ,1945-03-30,FRANK.LOPEZ@TEST.COM,+15551112222,FINISHED,null,null,2026-01-15T22:40:46.159Z
